# 🧪 Expérimentation MLOps (Docker, MLflow, MinIO & Discord)
Ce notebook démontre comment s'intègre toute l'architecture.

In [8]:
import sys
import os
import boto3
import mlflow
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Ajout du chemin parent pour importer src
sys.path.append(os.path.abspath('../'))
from src.utils.discord_notify import notify_discord

# 1. Création automatique du Bucket S3 (MinIO) si inexistant
s3 = boto3.client('s3', 
                  endpoint_url='http://minio:9000', 
                  aws_access_key_id='root', 
                  aws_secret_access_key='root123456789')
try:
    s3.create_bucket(Bucket='mlflow')
    print("Bucket 'mlflow' créé dans MinIO.")
except Exception:
    print("Bucket 'mlflow' déjà existant.")

# 2. Connexion au Tracking Server MLflow (Conteneur Docker)
mlflow.set_tracking_uri('http://mlflow:5000')


Bucket 'mlflow' déjà existant.


In [9]:
notify_discord("Début d'une session Jupyter dans l'environnement Dockerisé...", "info")

try:
    # Chargement des données
    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2)
    
    mlflow.set_experiment("Classification_Cancer")
    mlflow.sklearn.autolog()
    
    # Entraînement
    
    with mlflow.start_run() as run:
        notify_discord(f"Entraînement du modèle (Run ID: `{run.info.run_id}`)...", "info")
        
        clf = RandomForestClassifier(n_estimators=100, max_depth=5)
        clf.fit(X_train, y_train)
        
        # Evaluation
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)
        
        notify_discord(f"Entraînement terminé avec succès !\n**Précision :** `{acc:.4f}`\n*Le modèle a été sauvegardé dans MinIO via MLflow.*", "success")

except Exception as e:
    notify_discord(f"Alerte lors de l'entraînement :\n```python\n{str(e)}\n```", "error")
    raise e

2026/05/11 20:26:15 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.5.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/05/11 20:26:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run bold-wolf-897 at: http://mlflow:5000/#/experiments/1/runs/3fd03fe2a24548e1b630414cefd61f55
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [28]:
import requests
import json
from datetime import datetime
import logging
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("DiscordNotifier")

WEBHOOK_URL = os.getenv("DISCORD_WEBHOOK_URL", "https://discord.com/api/webhooks/1503504120072372436/tLMPJ6Erqsmz0Ok4awauPjcOKfP_eYMM9vsYcASIdZdPC_lgoggatfYZDYqVqe-UEG0n")

def notify_discord(message, run_id="N/A", metrics=None, params=None, level="success", task_name="Training"):
    """
    Notification Discord Optimisée - Style iTeam Univ
    """
    colors = {
        "info": 3447003, "success": 3066993, "warning": 16776960, "error": 15158332
    }
    
    # Construction des champs (Fields)
    # L'auteur et le statut sont en ligne
    fields = [
        {"name": "👤 Auteur", "value": "`ING. LABIADH LINA`", "inline": True},
        {"name": "⚙️ Statut", "value": "`Succès`" if level=="success" else "`Échec`", "inline": True},
        {"name": "🆔 Run ID", "value": f"`{run_id}`", "inline": False}
    ]

    # Ajout dynamique des hyperparamètres (n_estimators, etc.)
    if params:
        for k, v in params.items():
            fields.append({"name": k, "value": f"`{v}`", "inline": True})

    # Formatage des métriques
    metrics_text = "N/A"
    if metrics:
        metrics_text = "\n".join([f"**{k}:** `{v:.4f}`" for k, v in metrics.items()])
    
    fields.append({"name": "📊 Métriques", "value": metrics_text, "inline": False})
    fields.append({"name": "🔗 MLflow UI", "value": "[Accéder à l'interface](http://localhost:5000)", "inline": False})

    emoji = {"info": "ℹ️", "success": "✅", "error": "❌", "warning": "⚠️"}
    
    payload = {
        "username": "Bot-Notifier-MLOps",
        "avatar_url": "https://cdn-icons-png.flaticon.com/512/2103/2103633.png",
        "embeds": [{
            "title": f"{emoji.get(level, '➡️')} {task_name.upper()}",
            "description": message,
            "color": colors.get(level, 3447003),
            "fields": fields,
            "thumbnail": {"url": "https://cdn-icons-png.flaticon.com/512/4712/4712139.png"},
            "footer": {
                "text": "MLOps Lab • RandomForestClassifier • Cancer Dataset • ING. LINA"
            },
            "timestamp": datetime.utcnow().isoformat()
        }]
    }

    try:
        response = requests.post(WEBHOOK_URL, data=json.dumps(payload), headers={'Content-Type': 'application/json'}, timeout=5)
        response.raise_for_status()
    except Exception as e:
        logger.error(f"Erreur notification : {e}")

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import mlflow

# 1. Notification de démarrage
notify_discord("Pipeline MLOps démarré avec succès sur Docker.", level="info", task_name="Pipeline-Start")

try:
    # 2. Préparation des données
    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2)
    
    mlflow.set_experiment("Classification_Cancer")
    mlflow.sklearn.autolog()

    with mlflow.start_run() as run:
        # 3. Définition des hyperparamètres
        n_est, m_depth, m_feat = 100, 6, 3
        
        notify_discord(f"Entraînement en cours (Run: {run.info.run_id})...", level="info", task_name="Training")
        
        clf = RandomForestClassifier(n_estimators=n_est, max_depth=m_depth, max_features=m_feat)
        clf.fit(X_train, y_train)
        
        # 4. Évaluation
        preds = clf.predict(X_test)
        
        params_dict = {
            "n_estimators": n_est,
            "max_depth": m_depth,
            "max_features": m_feat
        }
        
        metrics_dict = {
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds),
            "Recall": recall_score(y_test, preds),
            "F1-Score": f1_score(y_test, preds)
        }
        
        # Log manuel pour garantir la présence dans MLflow UI
        mlflow.log_params(params_dict)
        mlflow.log_metrics(metrics_dict)
        
        # 5. Envoi du Rapport Final Automatique
        notify_discord(
            message="Le modèle a été entraîné et sauvegardé avec succès dans MinIO.",
            run_id=run.info.run_id,
            metrics=metrics_dict,
            params=params_dict,
            level="success",
            task_name="MLOps Lab - Rapport Automatique"
        )
    
except Exception as e:
    notify_discord(f"Erreur critique lors de l'exécution :\n`{str(e)}`", level="error", task_name="Failure")
    raise e

2026/05/11 22:05:45 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.5.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
2026/05/11 22:05:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run debonair-midge-958 at: http://mlflow:5000/#/experiments/1/runs/9d6e64dbea3f461ca40bb8fe96755ef1
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Notification de démarrage global
notify_discord("Pipeline MLOps démarré avec succès sur Docker.", metrics={"R2_Score": 0.8942}, level="success", task_name="Training")

try:
    # 1. Chargement des données
    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2)
    
    # 2. Configuration MLflow
    mlflow.set_experiment("Classification_Cancer")
    mlflow.sklearn.autolog()
    
    # 3. Entraînement et Evaluation
    with mlflow.start_run() as run:
        notify_discord(f"Début de l'entraînement...", level="info", task_name="ML-Pipeline")
            
        clf = RandomForestClassifier(n_estimators=100, max_depth=5)
        clf.fit(X_train, y_train)
            
        # Prédictions
        preds = clf.predict(X_test)
            
        # Calcul des métriques demandées
        metrics_dict = {
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds),
            "Recall": recall_score(y_test, preds),
            "F1-Score": f1_score(y_test, preds)
        }
            
        # Log manuel des métriques dans MLflow
        mlflow.log_metrics(metrics_dict)
            
        # Notification finale avec toutes les métriques et le Run ID
        notify_discord(
            message="Le modèle a été entraîné avec succès. Les performances sont disponibles ci-dessous.\nLe modèle a été sauvegardé dans MinIO via MLflow.", 
            run_id=run.info.run_id,
            metrics=metrics_dict, 
            level="success",
            task_name="Training-Complete"
        )
    
except Exception as e:
    # Notification en cas d'erreur
    notify_discord(f"Erreur critique détectée :\n{str(e)}", level="error", task_name="Failure")
    raise e

INFO:DiscordNotifier:Notification Discord envoyée avec succès : success
2026/05/11 21:49:35 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.5.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
INFO:DiscordNotifier:Notification Discord envoyée avec succès : info
2026/05/11 21:49:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
INFO:DiscordNotifier:Notification Discord envoyée avec succès : success


🏃 View run overjoyed-crow-740 at: http://mlflow:5000/#/experiments/1/runs/25084f2f676241f68c6e4965ac2d6d20
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Notification de démarrage
notify_discord("Pipeline MLOps démarré avec succès sur Docker.", level="success", task_name="Training")

try:
    data = load_breast_cancer()
    X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2)
    
    mlflow.set_experiment("Classification_Cancer")
    mlflow.sklearn.autolog()
    
    with mlflow.start_run() as run:
        notify_discord(f"Début de l'entraînement...", level="info", task_name="ML-Pipeline")
            
        # Modèle Random Forest
        clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
        clf.fit(X_train, y_train)
            
        preds = clf.predict(X_test)
        probs = clf.predict_proba(X_test)[:, 1] # Probabilités pour l'AUC-ROC
            
        # Métriques de classification
        metrics_dict = {
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds),
            "Recall": recall_score(y_test, preds),    # Critique pour le cancer
            "F1-Score": f1_score(y_test, preds),
            "AUC-ROC": roc_auc_score(y_test, probs)   # Performance globale du classifieur
        }
            
        mlflow.log_metrics(metrics_dict)
            
        notify_discord(
            message="Entraînement terminé. Le modèle RandomForest a été validé et sauvegardé.", 
            run_id=run.info.run_id,
            metrics=metrics_dict, 
            level="success",
            task_name="Model-Evaluation"
        )
    
except Exception as e:
    notify_discord(f"Erreur lors de l'entraînement :\n{str(e)}", level="error", task_name="Failure")
    raise e

INFO:DiscordNotifier:Notification Discord envoyée avec succès : success
2026/05/11 21:52:04 WARNING mlflow.utils.autologging_utils: MLflow sklearn autologging is known to be compatible with 1.5.0 <= scikit-learn, but the installed version is 1.3.1. If you encounter errors during autologging, try upgrading / downgrading scikit-learn to a compatible version, or try upgrading MLflow.
INFO:DiscordNotifier:Notification Discord envoyée avec succès : info
2026/05/11 21:52:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
INFO:DiscordNotifier:Notification Discord envoyée avec succès : success


🏃 View run defiant-hare-661 at: http://mlflow:5000/#/experiments/1/runs/b7d2c2aa396c49a5a03853720c98eb2d
🧪 View experiment at: http://mlflow:5000/#/experiments/1
